# <center>Spacy pipeline on Aioli annotations fetched from ArangoDB</center>

## Fetching data from ArangoDB

In [1]:
from arango import ArangoClient, database

client = ArangoClient(hosts="http://localhost:8529")
db : database.StandardDatabase = client.db("TEATIME", username="root", password="test")
annotations_collection : database.StandardCollection = db.collection("aioli_objects")

On va cherche toutes les annotations et on transforme le résultat en liste d'annotations. Chaque annotation est un dictionnaire

In [2]:
annotations_list = annotations_collection.all()
annotations_list = [_ for _ in annotations_list]

print(annotations_list[:5])

[{'_key': '0000f3158b8dfaf7', '_id': 'aioli_objects/0000f3158b8dfaf7', '_rev': '_lLdoJVK---', 'type': 'Region', 'name': 'CH00-14_DEPLAQUE01', 'project': '62e83c03ad7ae', 'layer': '6553de7468e2fea3', 'owner': 'MOE', 'material': 'purple', 'description': {'Commentaire': 'Zone déplaquée', 'Général': "Les travées flanquées à la croisée, n'ont pas pu être relevées en raison de la dépose, en cours, de l'échafaudage incendié et des vestiges non déblayés qui recouvrent les extrados. Un constat d'état complémentaire sera réalisé sur place lorsque l'accès sera dégagé.\n\n. Reins : Les reins de voûtes sont remplis de plomb fondu provenant de la couverture.\n. Voûtains: Pierres dégradées, fracturation ou déplaquage sur plusieurs centimètres à l'extrados des claveaux provoqué par le choc thermique de l'incendie, changement de couleurs indiquant des changements minéralogiques et de possibles modifications mécaniques. Lacunes ponctuelles: Zones affectées par la\nchute de bois de charpente, la durée et

## Using spacy on annotations

In [3]:
import spacy

### First with a single annotation in French

In [5]:
# En utilisant le petit modèle (30Mb~) pour chercher les entités nommées
nlp = spacy.load("fr_core_news_sm")
doc = nlp(annotations_list[0]['description']['Général'])

for ent in doc.ents:
    print (ent.text, ent.label_)

spacy.displacy.render(doc, style="ent")

Reins MISC
Voûtains LOC
Pierres LOC
Joints PER
Reprise LOC
Bo3 LOC
Tr3 LOC
Réalisation MISC


In [9]:
# Et ici en utilisant le modèle moyen

nlp = spacy.load("fr_core_news_md")
doc = nlp(annotations_list[0]['description']['Général'])

for ent in doc.ents:
    print (ent.text, ent.label_)

spacy.displacy.render(doc, style="ent")

Voûtains LOC
Pierres LOC
R MISC
Bo1 LOC
Bo3 LOC
Tr1 MISC
Tr3 MISC


In [4]:
# Et enfin le très gros modèle (572MB~)

nlp = spacy.load("fr_core_news_lg")
doc = nlp(annotations_list[0]['description']['Général'])

for ent in doc.ents:
    print (ent.text, ent.label_)

spacy.displacy.render(doc, style="ent")

/Users/marwan/Code/TEATIME_PhD_Experiments/.nlp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Voûtains LOC
Pierres PER
Bo1 LOC
Bo3 LOC
Tr1 LOC
Tr3 LOC


_On remarque qu'aucun des trois modèle n'est satisfaisant, dans la recherche d'entitées nommées liées au patrimoine. Il va donc falloir utiliser autre chose ou améliorer le modèle. On aimerai également ne voir que les liens avec des mots du corpus eventuellement?_

In [9]:
print("|TEXT|LEMMA|POS|TAG|DEP|SHAPE|ALPHA|STOP|")
print("|----|-----|---|---|---|-----|-----|----|")
for token in doc:
    print(f"|{token.text}|{token.lemma_}|{token.pos_}|{token.tag_}|{token.dep_}|{token.shape_}|{token.is_alpha}|{token.is_stop}|")

|TEXT|LEMMA|POS|TAG|DEP|SHAPE|ALPHA|STOP|
|----|-----|---|---|---|-----|-----|----|
|Les|le|DET|DET|det|Xxx|True|True|
|travées|travée|NOUN|NOUN|obl:mod|xxxx|True|False|
|flanquées|flanquer|VERB|VERB|acl|xxxx|True|False|
|à|à|ADP|ADP|case|x|True|True|
|la|le|DET|DET|det|xx|True|True|
|croisée|croisée|NOUN|NOUN|obl:arg|xxxx|True|False|
|,|,|PUNCT|PUNCT|punct|,|False|False|
|n'|ne|ADV|ADV|advmod|x'|False|True|
|ont|avoir|AUX|AUX|aux:tense|xxx|True|True|
|pas|pas|ADV|ADV|advmod|xxx|True|True|
|pu|pouvoir|VERB|VERB|ROOT|xx|True|True|
|être|être|AUX|AUX|aux:pass|xxxx|True|True|
|relevées|relever|VERB|VERB|xcomp|xxxx|True|False|
|en|en|ADP|ADP|case|xx|True|True|
|raison|raison|NOUN|NOUN|obl:mod|xxxx|True|False|
|de|de|ADP|ADP|case|xx|True|True|
|la|le|DET|DET|det|xx|True|True|
|dépose|dépose|NOUN|NOUN|nmod|xxxx|True|False|
|,|,|PUNCT|PUNCT|punct|,|False|False|
|en|en|ADP|ADP|case|xx|True|True|
|cours|cours|NOUN|NOUN|obl:mod|xxxx|True|False|
|,|,|PUNCT|PUNCT|punct|,|False|False|
|de|de|ADP|AD

## Tableau reprenant chaque token de notre texte

|TEXT|LEMMA|POS|TAG|DEP|SHAPE|ALPHA|STOP|
|----|-----|---|---|---|-----|-----|----|
|Les|le|DET|DET|det|Xxx|True|True|
|travées|travée|NOUN|NOUN|obl:mod|xxxx|True|False|
|flanquées|flanquer|VERB|VERB|acl|xxxx|True|False|
|à|à|ADP|ADP|case|x|True|True|
|la|le|DET|DET|det|xx|True|True|
|croisée|croisée|NOUN|NOUN|obl:arg|xxxx|True|False|
|,|,|PUNCT|PUNCT|punct|,|False|False|
|n'|ne|ADV|ADV|advmod|x'|False|True|
|ont|avoir|AUX|AUX|aux:tense|xxx|True|True|
|pas|pas|ADV|ADV|advmod|xxx|True|True|
|pu|pouvoir|VERB|VERB|ROOT|xx|True|True|
|être|être|AUX|AUX|aux:pass|xxxx|True|True|
|relevées|relever|VERB|VERB|xcomp|xxxx|True|False|
|en|en|ADP|ADP|case|xx|True|True|
|raison|raison|NOUN|NOUN|obl:mod|xxxx|True|False|
|de|de|ADP|ADP|case|xx|True|True|
|la|le|DET|DET|det|xx|True|True|
|dépose|dépose|NOUN|NOUN|nmod|xxxx|True|False|
|,|,|PUNCT|PUNCT|punct|,|False|False|
|en|en|ADP|ADP|case|xx|True|True|
|cours|cours|NOUN|NOUN|obl:mod|xxxx|True|False|
|,|,|PUNCT|PUNCT|punct|,|False|False|
|de|de|ADP|ADP|case|xx|True|True|
|l'|le|DET|DET|det|x'|False|True|
|échafaudage|échafaudage|NOUN|NOUN|nmod|xxxx|True|False|
|incendié|incendier|VERB|VERB|acl|xxxx|True|False|
|et|et|CCONJ|CCONJ|cc|xx|True|True|
|des|de|ADP|ADP|case|xxx|True|True|
|vestiges|vestige|NOUN|NOUN|conj|xxxx|True|False|
|non|non|ADV|ADV|advmod|xxx|True|False|
|déblayés|déblayer|VERB|VERB|acl|xxxx|True|False|
|qui|qui|PRON|PRON|nsubj|xxx|True|True|
|recouvrent|recouvrir|VERB|VERB|acl:relcl|xxxx|True|False|
|les|le|DET|DET|det|xxx|True|True|
|extrados|extrados|NOUN|NOUN|obj|xxxx|True|False|
|.|.|PUNCT|PUNCT|punct|.|False|False|
|Un|un|DET|DET|det|Xx|True|True|
|constat|constat|NOUN|NOUN|nsubj:pass|xxxx|True|False|
|d'|de|ADP|ADP|case|x'|False|True|
|état|état|NOUN|NOUN|nmod|xxxx|True|False|
|complémentaire|complémentaire|ADJ|ADJ|amod|xxxx|True|False|
|sera|être|AUX|AUX|aux:pass|xxxx|True|True|
|réalisé|réaliser|VERB|VERB|ROOT|xxxx|True|False|
|sur|sur|ADP|ADP|case|xxx|True|True|
|place|place|NOUN|NOUN|obl:mod|xxxx|True|False|
|lorsque|lorsque|ADP|ADP|mark|xxxx|True|True|
|l'|le|DET|DET|det|x'|False|True|
|accès|accès|NOUN|NOUN|nsubj:pass|xxxx|True|False|
|sera|être|AUX|AUX|aux:pass|xxxx|True|True|
|dégagé|dégager|VERB|VERB|advcl|xxxx|True|False|
|.|.|PUNCT|PUNCT|punct|.|False|False|
|.|.|PUNCT|PUNCT|punct|.|False|False|
|Reins|Reins|PROPN|PROPN|ROOT|Xxxxx|True|False|
|:|:|PUNCT|PUNCT|punct|:|False|False|
|Les|le|DET|DET|det|Xxx|True|True|
|reins|rein|NOUN|NOUN|nsubj:pass|xxxx|True|False|
|de|de|ADP|ADP|case|xx|True|True|
|voûtes|voûte|NOUN|NOUN|nmod|xxxx|True|False|
|sont|être|AUX|AUX|aux:pass|xxxx|True|True|
|remplis|remplir|VERB|VERB|ROOT|xxxx|True|False|
|de|de|ADP|ADP|case|xx|True|True|
|plomb|plomb|NOUN|NOUN|obl:arg|xxxx|True|False|
|fondu|fondre|VERB|VERB|acl|xxxx|True|False|
|provenant|provenir|VERB|VERB|xcomp|xxxx|True|False|
|de|de|ADP|ADP|case|xx|True|True|
|la|le|DET|DET|det|xx|True|True|
|couverture|couverture|NOUN|NOUN|obl:arg|xxxx|True|False|
|.|.|PUNCT|PUNCT|punct|.|False|False|
|.|.|PUNCT|PUNCT|punct|.|False|False|
|Voûtains|Voûtains|PROPN|PROPN|det|Xxxxx|True|False|
|:|:|PUNCT|PUNCT|punct|:|False|False|
|Pierres|pierre|NOUN|NOUN|ROOT|Xxxxx|True|False|
|dégradées|dégrader|VERB|VERB|amod|xxxx|True|False|
|,|,|PUNCT|PUNCT|punct|,|False|False|
|fracturation|fracturation|NOUN|NOUN|conj|xxxx|True|False|
|ou|ou|CCONJ|CCONJ|cc|xx|True|True|
|déplaquage|déplaquage|PROPN|PROPN|conj|xxxx|True|False|
|sur|sur|ADP|ADP|case|xxx|True|True|
|plusieurs|plusieurs|DET|DET|det|xxxx|True|True|
|centimètres|centimètre|NOUN|NOUN|nmod|xxxx|True|False|
|à|à|ADP|ADP|case|x|True|True|
|l'|le|DET|DET|det|x'|False|True|
|extrados|extrados|NOUN|NOUN|nmod|xxxx|True|False|
|des|de|ADP|ADP|case|xxx|True|True|
|claveaux|claveau|NOUN|NOUN|nmod|xxxx|True|False|
|provoqué|provoquer|VERB|VERB|acl|xxxx|True|False|
|par|par|ADP|ADP|case|xxx|True|True|
|le|le|DET|DET|det|xx|True|True|
|choc|choc|NOUN|NOUN|obl:agent|xxxx|True|False|
|thermique|thermique|ADJ|ADJ|amod|xxxx|True|False|
|de|de|ADP|ADP|case|xx|True|True|
|l'|le|DET|DET|det|x'|False|True|
|incendie|incendie|NOUN|NOUN|nmod|xxxx|True|False|
|,|,|PUNCT|PUNCT|punct|,|False|False|
|changement|changement|NOUN|NOUN|obj|xxxx|True|False|
|de|de|ADP|ADP|case|xx|True|True|
|couleurs|couleur|NOUN|NOUN|nmod|xxxx|True|False|
|indiquant|indiquer|VERB|VERB|acl|xxxx|True|False|
|des|un|DET|DET|det|xxx|True|True|
|changements|changement|NOUN|NOUN|obj|xxxx|True|False|
|minéralogiques|minéralogique|ADJ|ADJ|amod|xxxx|True|False|
|et|et|CCONJ|CCONJ|cc|xx|True|True|
|de|de|DET|DET|det|xx|True|True|
|possibles|possible|ADJ|ADJ|amod|xxxx|True|True|
|modifications|modification|NOUN|NOUN|conj|xxxx|True|False|
|mécaniques|mécanique|ADJ|ADJ|amod|xxxx|True|False|
|.|.|PUNCT|PUNCT|punct|.|False|False|
|Lacunes|lacune|ADJ|ADJ|ROOT|Xxxxx|True|False|
|ponctuelles|ponctuel|ADJ|ADJ|ROOT|xxxx|True|False|
|:|:|PUNCT|PUNCT|punct|:|False|False|
|Zones|zone|NOUN|NOUN|ROOT|Xxxxx|True|False|
|affectées|affecter|VERB|VERB|acl|xxxx|True|False|
|par|par|ADP|ADP|case|xxx|True|True|
|la|le|DET|DET|det|xx|True|True|
|chute|chute|NOUN|NOUN|obl:agent|xxxx|True|False|
|de|de|ADP|ADP|case|xx|True|True|
|bois|bois|NOUN|NOUN|nmod|xxxx|True|False|
|de|de|ADP|ADP|case|xx|True|True|
|charpente|charpente|NOUN|NOUN|nmod|xxxx|True|False|
|,|,|PUNCT|PUNCT|punct|,|False|False|
|la|le|DET|DET|det|xx|True|True|
|durée|durée|NOUN|NOUN|nmod|xxxx|True|False|
|et|et|CCONJ|CCONJ|cc|xx|True|True|
|la|le|DET|DET|det|xx|True|True|
|puissance|puissance|NOUN|NOUN|conj|xxxx|True|False|
|de|de|ADP|ADP|case|xx|True|True|
|l'|le|DET|DET|det|x'|False|True|
|incendie|incendie|NOUN|NOUN|nmod|xxxx|True|False|
|.|.|PUNCT|PUNCT|punct|.|False|False|
|Zones|zone|NOUN|NOUN|ROOT|Xxxxx|True|False|
|fragilisées|fragiliser|ADJ|ADJ|amod|xxxx|True|False|
|au|au|ADP|ADP|case|xx|True|True|
|droit|droit|NOUN|NOUN|nmod|xxxx|True|False|
|des|de|ADP|ADP|case|xxx|True|True|
|pierres|pierre|NOUN|NOUN|nmod|xxxx|True|False|
|lacunaires|lacunaire|ADJ|ADJ|amod|xxxx|True|False|
|.|.|PUNCT|PUNCT|punct|.|False|False|
|Joints|joint|NOUN|NOUN|ROOT|Xxxxx|True|False|
|dégradés|dégrader|VERB|VERB|acl|xxxx|True|False|
|et|et|CCONJ|CCONJ|cc|xx|True|True|
|évidés|évider|VERB|VERB|conj|xxxx|True|False|
|en|en|ADP|ADP|case|xx|True|True|
|profondeur|profondeur|NOUN|NOUN|obl:mod|xxxx|True|False|
|,|,|PUNCT|PUNCT|punct|,|False|False|
|phénomène|phénomène|NOUN|NOUN|appos|xxxx|True|False|
|de|de|ADP|ADP|case|xx|True|True|
|décarbonatation|décarbonatation|NOUN|NOUN|nmod|xxxx|True|False|
|des|de|ADP|ADP|case|xxx|True|True|
|joints|joint|NOUN|NOUN|nmod|xxxx|True|False|
|à|à|ADP|ADP|case|x|True|True|
|la|le|DET|DET|det|xx|True|True|
|chaux|chaux|NOUN|NOUN|nmod|xxxx|True|False|
|.|.|PUNCT|PUNCT|punct|.|False|False|
|Chape|chape|NOUN|NOUN|ROOT|Xxxxx|True|False|
|de|de|ADP|ADP|case|xx|True|True|
|protection|protection|NOUN|NOUN|nmod|xxxx|True|False|
|irrégulière|irrégulier|ADJ|ADJ|amod|xxxx|True|False|
|et|et|CCONJ|CCONJ|cc|xx|True|True|
|déplaquée|déplaquée|NOUN|NOUN|conj|xxxx|True|False|
|à|à|ADP|ADP|case|x|True|True|
|près|près|ADV|ADV|advmod|xxxx|True|True|
|de|de|ADP|ADP|dep|xx|True|True|
|80|80|NUM|NUM|nummod|dd|False|False|
|%|pourcent|NOUN|NOUN|nmod|%|False|False|
|,|,|PUNCT|PUNCT|punct|,|False|False|
|pouvant|pouvoir|VERB|VERB|acl|xxxx|True|False|
|s'|se|PRON|PRON|expl:pass|x'|False|True|
|expliquer|expliquer|VERB|VERB|xcomp|xxxx|True|False|
|par|par|ADP|ADP|case|xxx|True|True|
|la|le|DET|DET|det|xx|True|True|
|désagrégation|désagrégation|NOUN|NOUN|obl:arg|xxxx|True|False|
|du|de|ADP|ADP|case|xx|True|True|
|mortier|mortier|NOUN|NOUN|nmod|xxxx|True|False|
|à|à|ADP|ADP|case|x|True|True|
|cause|cause|NOUN|NOUN|nmod|xxxx|True|False|
|de|de|ADP|ADP|case|xx|True|True|
|l'|le|DET|DET|det|x'|False|True|
|humidité|humidité|NOUN|NOUN|nmod|xxxx|True|False|
|ou|ou|CCONJ|CCONJ|cc|xx|True|True|
|des|un|DET|DET|det|xxx|True|True|
|campagnes|campagne|NOUN|NOUN|conj|xxxx|True|False|
|de|de|ADP|ADP|case|xx|True|True|
|restaurations|restauration|NOUN|NOUN|nmod|xxxx|True|False|
|successives|successif|ADJ|ADJ|amod|xxxx|True|False|
|des|de|ADP|ADP|case|xxx|True|True|
|voûtains|voûtain|NOUN|NOUN|nmod|xxxx|True|False|
|sans|sans|ADP|ADP|mark|xxxx|True|True|
|remise|remettre|VERB|VERB|advcl|xxxx|True|False|
|en|en|ADP|ADP|case|xx|True|True|
|oeuvre|oeuvre|NOUN|NOUN|obl:arg|xxxx|True|False|
|de|de|ADP|ADP|case|xx|True|True|
|la|le|DET|DET|det|xx|True|True|
|couche|couche|NOUN|NOUN|obl:arg|xxxx|True|False|
|d'|de|ADP|ADP|case|x'|False|True|
|enduit|enduit|NOUN|NOUN|nmod|xxxx|True|False|
|.|.|PUNCT|PUNCT|punct|.|False|False|
|Absence|absence|NOUN|NOUN|ROOT|Xxxxx|True|False|
|de|de|ADP|ADP|case|xx|True|True|
|chape|chape|NOUN|NOUN|nmod|xxxx|True|False|
|de|de|ADP|ADP|case|xx|True|True|
|protectection|protectection|NOUN|NOUN|nmod|xxxx|True|False|
|sur|sur|ADP|ADP|case|xxx|True|True|
|le|le|DET|DET|det|xx|True|True|
|voutain|voutain|ADJ|ADJ|amod|xxxx|True|False|
|central|central|ADJ|ADJ|nmod|xxxx|True|False|
|des|de|ADP|ADP|case|xxx|True|True|
|travées|travée|NOUN|NOUN|nmod|xxxx|True|False|
|21|21|NOUN|NOUN|nmod|dd|False|False|
|-|-|NOUN|NOUN|punct|-|False|False|
|22|22|NOUN|NOUN|ROOT|dd|False|False|
|.|.|PUNCT|PUNCT|ROOT|.|False|False|
|Reprise|reprise|NOUN|NOUN|ROOT|Xxxxx|True|False|
|(|(|PUNCT|PUNCT|punct|(|False|False|
|R|r|NOUN|NOUN|nmod|X|True|False|
|)|)|PUNCT|PUNCT|punct|)|False|False|
|du|de|ADP|ADP|case|xx|True|True|
|voutain|voutain|PROPN|PROPN|nmod|xxxx|True|False|
|lors|lors|ADV|ADV|dep|xxxx|True|True|
|d'|de|ADP|ADP|case|x'|False|True|
|une|un|DET|DET|det|xxx|True|True|
|restauration|restauration|NOUN|NOUN|obl:arg|xxxx|True|False|
|ancienne|ancien|ADJ|ADJ|amod|xxxx|True|False|
|.|.|PUNCT|PUNCT|punct|.|False|False|
|Elément|elément|NOUN|NOUN|ROOT|Xxxxx|True|False|
|bois|boi|ADJ|ADJ|amod|xxxx|True|False|
|inséré|insérer|VERB|VERB|acl|xxxx|True|False|
|dans|dans|ADP|ADP|case|xxxx|True|True|
|la|le|DET|DET|det|xx|True|True|
|maçonnerie|maçonnerie|NOUN|NOUN|obl:mod|xxxx|True|False|
|(|(|PUNCT|PUNCT|punct|(|False|False|
|Bo1|bo1|NOUN|NOUN|appos|Xxd|False|False|
|à|à|ADP|ADP|case|x|True|True|
|Bo3|bo3|PRON|PRON|nmod|Xxd|False|False|
|)|)|PUNCT|PUNCT|punct|)|False|False|
|.|.|PUNCT|PUNCT|punct|.|False|False|
|Trous|trou|NOUN|NOUN|ROOT|Xxxxx|True|False|
|de|de|ADP|ADP|case|xx|True|True|
|la|le|DET|DET|det|xx|True|True|
|voûte|voûte|NOUN|NOUN|nmod|xxxx|True|False|
|,|,|PUNCT|PUNCT|punct|,|False|False|
|voussoirs|voussoir|NOUN|NOUN|amod|xxxx|True|False|
|lacunaires|lacunaire|ADJ|ADJ|amod|xxxx|True|False|
|(|(|PUNCT|PUNCT|punct|(|False|False|
|Tr1|tr1|NOUN|NOUN|nmod|Xxd|False|False|
|à|à|ADP|ADP|case|x|True|True|
|Tr3|Tr3|PROPN|PROPN|nmod|Xxd|False|False|
|)|)|PUNCT|PUNCT|punct|)|False|False|
|.|.|PUNCT|PUNCT|punct|.|False|False|
|Consolidation|consolidation|NOUN|NOUN|ROOT|Xxxxx|True|False|
|:|:|PUNCT|PUNCT|punct|:|False|False|
|Réalisation|réalisation|NOUN|NOUN|nmod|Xxxxx|True|False|
|d'|de|ADP|ADP|case|x'|False|True|
|un|un|DET|DET|det|xx|True|True|
|cataplasme|cataplasme|NOUN|NOUN|nmod|xxxx|True|False|
|constitué|constituer|VERB|VERB|acl|xxxx|True|False|
|de|de|ADP|ADP|case|xx|True|True|
|plâtre|plâtre|NOUN|NOUN|obl:arg|xxxx|True|False|
|et|et|CCONJ|CCONJ|cc|xx|True|True|
|filasse|filasse|NOUN|NOUN|conj|xxxx|True|False|
|autour|autour|ADV|ADV|advmod|xxxx|True|False|
|des|de|ADP|ADP|case|xxx|True|True|
|franges|frange|NOUN|NOUN|obl:arg|xxxx|True|False|
|des|de|ADP|ADP|case|xxx|True|True|
|trous|trou|NOUN|NOUN|nmod|xxxx|True|False|
|afin|afin|ADV|ADV|ROOT|xxxx|True|True|
|de|de|ADP|ADP|case|xx|True|True|
|sécuriser|sécuriser|VERB|VERB|obl:arg|xxxx|True|False|
|les|le|DET|DET|det|xxx|True|True|
|voûtes|voûte|NOUN|NOUN|obj|xxxx|True|False|
|provisoirement|provisoirement|ADV|ADV|amod|xxxx|True|False|
|et|et|CCONJ|CCONJ|cc|xx|True|True|
|de|de|ADP|ADP|mark|xx|True|True|
|permettre|permettre|VERB|VERB|conj|xxxx|True|False|
|leur|son|DET|DET|det|xxxx|True|True|
|mise|mise|NOUN|NOUN|obj|xxxx|True|False|
|sur|sur|ADP|ADP|case|xxx|True|True|
|cintre|cintre|NOUN|NOUN|nmod|xxxx|True|False|
|.|.|PUNCT|PUNCT|ROOT|.|False|False|